# 🌍 Word2Vec: "Dimmi con chi vai e ti dirò chi sei"

Fino ad ora abbiamo trattato le parole come entità isolate (BoW, TF-IDF). Per un computer, in quei modelli, le parole **"Pizza"** e **"Hamburger"** sono diverse tanto quanto **"Pizza"** e **"Astronave"**. Non esiste una nozione di "vicinanza" semantica.

**Word2Vec** cambia paradigma: smette di contare le parole e inizia a studiare le loro **relazioni**.

---

## 1. L'Intuizione: L'Ipotesi Distribuzionale
Il cuore di Word2Vec è un concetto linguistico potentissimo: *"Conoscerai una parola dalla compagnia che frequenta"*. 

Immaginate di leggere questa frase in una lingua sconosciuta:
> "Oggi a pranzo ho mangiato un saporito **glifron** con le patatine."

Anche se non sapete cos'è un *glifron*, il contesto vi dice molto:
* È qualcosa di commestibile.
* È probabilmente un piatto salato.
* Si accompagna bene alle patatine.

**Word2Vec impara esattamente così:** assegna un significato alle parole osservando i termini che compaiono sistematicamente nel loro vicinato (contesto).

---

## 2. Dai Vettori Sparsi ai Vettori Densi
Word2Vec rivoluziona il modo in cui rappresentiamo i dati, passando da matrici enormi e vuote a vettori compatti e ricchi di informazione.

| Caratteristica | BoW / TF-IDF | Word2Vec (Embeddings) |
| :--- | :--- | :--- |
| **Dimensione** | Enorme (pari al vocabolario) | Piccola (solitamente 100-300) |
| **Valori** | Quasi tutti zeri (**Sparsi**) | Numeri decimali (**Densi**) |
| **Relazioni** | Nessuna (parole indipendenti) | Spaziali (vicinanza = similitudine) |
| **Significato** | Basato sulla frequenza pura | Basato sul contesto e l'uso |

---

## 3. Le due architetture di Word2Vec
Il modello impara le relazioni tra le parole attraverso due possibili strategie di addestramento:

### A) CBOW (Continuous Bag of Words)
Il modello osserva le parole circostanti (il contesto) e cerca di **prevedere la parola mancante** al centro.
* *Input:* "Il [???] insegue il topo" $\rightarrow$ *Target:* "gatto".

### B) Skip-gram
Il modello osserva una parola specifica e cerca di **prevedere il contesto** più probabile che la circonda.
* *Input:* "gatto" $\rightarrow$ *Target:* ["insegue", "topo", "miagola"].

![Architetture Word2Vec: CBOW vs Skip-gram](path/to/your/image_w2v.png)

---

## 4. La Geometria del Significato
In Word2Vec, ogni parola è un punto in uno spazio ad $N$ dimensioni. La "similitudine" tra due termini non è più un'opinione, ma una misura matematica chiamata **Cosine Similarity**:

$$\text{sim}(\vec{A}, \vec{B}) = \frac{\vec{A} \cdot \vec{B}}{\|\vec{A}\| \|\vec{B}\|}$$

### L'Algebra delle Parole
La vera magia degli embeddings è che le direzioni nello spazio vettoriale corrispondono a relazioni semantiche. È possibile eseguire operazioni aritmetiche tra concetti:

> $$V_{Re} - V_{Uomo} + V_{Donna} \approx V_{Regina}$$

Se dal vettore di "Re" sottraiamo la componente maschile e aggiungiamo quella femminile, il risultato ci porta matematicamente vicini al vettore di "Regina". Altri esempi includono le capitali ($V_{Italia} - V_{Roma} + V_{Parigi} \approx V_{Francia}$) o i tempi verbali.

---

> **Punto chiave per la lezione:** > Word2Vec non "capisce" le parole come un essere umano. È un campione di **statistica sociale**: sa che certe parole "frequentano gli stessi posti" e ne deduce che debbano avere significati simili. Questa capacità di comprimere il significato in vettori densi è la base di quasi tutti i modelli moderni di Deep Learning e dei Large Language Models (LLM).

# Parte Pratica

In [ ]:
# ==============================================================================
# BLOCCO 1: INSTALLAZIONE E CARICAMENTO DEL MODELLO
#
# In questa prima parte, diciamo a Google Colab di installare
# gli strumenti che ci servono e di scaricare il modello Word2Vec.
# Un "modello" in questo caso è come un gigantesco dizionario
# che conosce le relazioni tra milioni di parole.
# ==============================================================================

# Il comando "!" dice a Colab di eseguire un comando nel "terminale"
# "pip install" è il comando standard di Python per installare nuovi "pacchetti" (strumenti)
# "gensim" è il nome del pacchetto che sa come gestire e usare Word2Vec.
# "sklearn" (scikit-learn) è un pacchetto per fare "Machine Learning" (come il clustering)
print("--- 1. Installo gli strumenti necessari (gensim e sklearn)... ---")
!pip install gensim scikit-learn

# Ora importiamo gli strumenti che useremo.
# "import" è come dire "metti in questa pagina lo strumento che ti chiedo".

# "gensim.downloader" è lo strumento specifico per SCARICARE modelli pre-addestrati
import gensim.downloader
# "numpy" è uno strumento potentissimo per lavorare con i numeri (i vettori sono liste di numeri)
import numpy as np
# Da "sklearn" importiamo solo i 2 algoritmi che ci servono
from sklearn.cluster import KMeans  # Per il clustering (creare gruppi)
from sklearn.linear_model import LogisticRegression  # Per la classificazione (decidere "sì/no")
import matplotlib.pyplot as plt # Per disegnare i grafici
from sklearn.cluster import KMeans               # Algoritmo di Clustering
from sklearn.decomposition import PCA            # Per ridurre le dimensioni (per il grafico)
import pandas as pd                              # Per lavorare con tabelle
from sklearn.model_selection import train_test_split # Per dividere i dati (training/test)
from sklearn.metrics import accuracy_score     # Per misurare l'accuratezza

In [ ]:
print("\n--- 2. Scarico il modello Word2Vec... ---")
print("(Questo passaggio è LUNGO e richiede qualche minuto. Fallo solo una volta!)")

# "gensim.downloader.load" è il comando per scaricare e caricare il modello.
# 'word2vec-google-news-300' è il nome del modello famoso addestrato da Google.
# È un file enorme (circa 1.6 GB)!
# Salviamo tutto in una "variabile" chiamata "modello_w2v"
# Una variabile è solo un'etichetta, un nome per contenere qualcosa.
modello_w2v = gensim.downloader.load('word2vec-google-news-300')

print("\n--- 3. Modello caricato! Pronto a lavorare. ---")

In [ ]:
# Esploriamo il modello
print(f"Dimensione del vocabolario: {len(modello_w2v.key_to_index)}")
print(f"Dimensione dei vettori (embeddings): {modello_w2v.vector_size}")

In [ ]:
#mostriamo il vettore di una parola
print(modello_w2v['computer'])

In [ ]:
# ==============================================================================
# BLOCCO 2: OPERAZIONI DI BASE E ALGEBRA VETTORIALE
#
# Ora usiamo il modello! Vediamo cosa "sa" fare.
# La cosa più semplice è trovare parole simili.
# ==============================================================================

print("\n\n--- INIZIO CASO D'USO 1: SIMILARITÀ E ALGEBRA ---")

# --- Esempio 1: Trovare parole simili ---
# Chiediamo al modello: quali sono le 5 parole più simili a 'computer'?
# "most_similar" significa "più simile"
print("\nParole più simili a 'computer':")
simili_a_computer = modello_w2v.most_similar('computer', topn=5)
print(simili_a_computer)

In [ ]:
# --- Esempio 2: Misurare la similarità ---
# Possiamo chiedere "quanto" due parole sono simili (con un punteggio da -1 a 1)
print("\nMisura la similarità tra 'cane' e 'gatto':")
sim_cane_gatto = modello_w2v.similarity('dog', 'cat')
print(sim_cane_gatto)  # Ci aspettiamo un numero alto (es. 0.8)

print("\nMisura la similarità tra 'cane' e 'tavolo':")
sim_cane_tavolo = modello_w2v.similarity('dog', 'table')
print(sim_cane_tavolo) # Ci aspettiamo un numero basso (es. 0.1)

In [ ]:
# --- Esempio 3: L'algebra delle parole (la parte "magica") ---
# Possiamo fare somme e sottrazioni.
# L'esempio classico: Re - Uomo + Donna = ??? (Dovrebbe dare 'Regina')

# "positive" = le parole da SOMMARE (king, woman)
# "negative" = le parole da SOTTRARRE (man)
print("\nAnalisi: 're' - 'uomo' + 'donna' = ???")
risultato = modello_w2v.most_similar(positive=['king', 'woman'], negative=['man'], topn=1)
print(risultato)

In [ ]:
# Proviamo un esempio geografico:
# 'Parigi' - 'Francia' + 'Italia' = ??? (Dovrebbe dare 'Roma')
print("\nAnalisi: 'Parigi' - 'Francia' + 'Italia' = ???")
risultato = modello_w2v.most_similar(positive=['Paris', 'Italy'], negative=['France'], topn=1)
print(risultato)

In [ ]:
# Proviamo a giocare un po' 
print("\nAnalisi: 'Cristiano_Ronaldo' - 'Portogallo' + 'Argentina' = ???")
risultato = modello_w2v.most_similar(positive=['Cristiano_Ronaldo', 'Argentina'], negative=['Portugal'], topn=3)
print(risultato)

In [ ]:
print("\n\n--- INIZIO CASO D'USO 2 (A): CLUSTERING ---")

# 1. Lista di parole di 3 categorie diverse
lista_parole = [
    'dog', 'cat', 'lion', 'tiger', 'puppy', 'kitten',  # Animali
    'apple', 'banana', 'orange', 'strawberry', 'fruit', # Frutta
    'car', 'bicycle', 'train', 'plane', 'truck', 'ship' # Veicoli
]
print(f"La nostra lista di parole è: {lista_parole}")

# 2. Estraiamo i vettori (la lista di 300 numeri) per ogni parola
# (Usiamo una "list comprehension", un modo veloce in Python per creare una lista)
lista_vettori = [modello_w2v[parola] for parola in lista_parole]

# 3. Creiamo e addestriamo l'algoritmo K-Means
# "n_clusters=3" -> "Trova 3 gruppi"
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(lista_vettori)

# 4. Mostriamo i risultati
print("\nRisultati del clustering (testuali):")
# "kmeans.labels_" contiene i numeri dei gruppi (0, 1, o 2) per ogni parola
labels = kmeans.labels_
for i in range(len(lista_parole)):
    print(f"Parola: '{lista_parole[i]}' -> assegnata al Gruppo: {labels[i]}")

In [ ]:
print("\n\n--- INIZIO CASO D'USO 2 (B): VISUALIZZAZIONE CLUSTERING ---")

# 1. Il problema: i nostri vettori hanno 300 dimensioni (numeri).
#    Non possiamo disegnare in 300 dimensioni!
#    Dobbiamo "comprimere" i vettori in 2 dimensioni (un punto X e un punto Y).
#    Usiamo un algoritmo chiamato PCA (Principal Component Analysis).

print("Comprimo i vettori da 300 a 2 dimensioni (X, Y) usando PCA...")
pca = PCA(n_components=2, random_state=42) # "Voglio 2 componenti (dimensioni)"
vettori_2d = pca.fit_transform(lista_vettori) # "Comprimi!"

# "vettori_2d" ora contiene le coordinate (X, Y) per ogni parola.

# 2. Creiamo il grafico
print("Disegno il grafico...")
plt.figure(figsize=(10, 7)) # Imposta la dimensione del grafico (larghezza 10, altezza 7)

# "plt.scatter" è il comando per disegnare un grafico a "punti"
# x=vettori_2d[:, 0] -> Prendi tutti i valori della prima colonna (la X)
# y=vettori_2d[:, 1] -> Prendi tutti i valori della seconda colonna (la Y)
# c=labels -> Colora ogni punto in base al gruppo (label) che gli ha dato K-Means
plt.scatter(vettori_2d[:, 0], vettori_2d[:, 1], c=labels, cmap='viridis', s=100)

# 3. Aggiungiamo le etichette (i nomi delle parole) sul grafico
# Facciamo un ciclo per scrivere ogni parola vicino al suo punto
for i, parola in enumerate(lista_parole):
    plt.annotate(parola, (vettori_2d[i, 0] + 0.02, vettori_2d[i, 1]))

# 4. Aggiungiamo titoli e mostriamo il grafico
plt.title('Visualizzazione Clustering Word2Vec (con PCA)')
plt.xlabel('Dimensione 1 (PCA)')
plt.ylabel('Dimensione 2 (PCA)')
plt.grid(True) # Aggiunge una griglia per leggibilità
plt.show() # Questo comando "stampa" il grafico a schermo

In [ ]:
# ==============================================================================
# 🎯 BLOCCO 2: CARICAMENTO E PULIZIA DEI DATI (Toxicity)
#
# Obiettivo: Caricare i dati "veri" da un URL e pulirli
# per renderli utilizzabili.
# ==============================================================================

print("\n\n--- 4. Carico il dataset 'Toxicity' dall'URL... ---")
# Questo è un file CSV (un foglio di calcolo) che sta su Internet
url_dataset = "https://raw.githubusercontent.com/t-davidson/hate-speech-and-offensive-language/master/data/labeled_data.csv"

# Usiamo "pandas" (pd) per leggere il file CSV e trasformarlo in una tabella
df_completo = pd.read_csv(url_dataset)

print(f"Caricate {len(df_completo)} righe. Ecco un'anteprima dei dati 'grezzi':")
# .head() ci fa vedere le prime 5 righe
df_completo.head()

In [ ]:
print("\n--- 5. Pulizia e Semplificazione dei Dati ---")
# I dati grezzi sono confusi. Hanno 3 classi (0, 1, 2) e colonne inutili.
# Noi vogliamo una classificazione semplice: 0 (NON TOSSICO) vs 1 (TOSSICO)

# La colonna 'class' ha 3 valori:
# 0 = Hate Speech (Odio)
# 1 = Offensive Language (Offensivo)
# 2 = Neither (Nessuno dei due)

# Creiamo una nuova colonna 'is_toxic'.
# Sarà 1 (Tossico) se la classe è 0 OPPURE 1.
# Sarà 0 (Non Tossico) se la classe è 2.
# ".apply(lambda x: ...)" è un modo per applicare questa regola a ogni riga
df_completo['is_toxic'] = df_completo['class'].apply(lambda x: 1 if x < 2 else 0)

# Ora selezioniamo SOLO le colonne che ci servono:
# - Il testo del tweet
# - La nostra nuova etichetta 'is_toxic'
df_pulito = df_completo[['tweet', 'is_toxic']]

# Rinominiamo 'tweet' in 'comment_text' per chiarezza
df_pulito = df_pulito.rename(columns={'tweet': 'comment_text'})

print("\nDati puliti e pronti. Ecco un'anteprima:")
df_pulito.head()

In [ ]:
# ==============================================================================
# ⚖️ BLOCCO 3: GESTIONE DELLO SQUILIBRIO (Balancing)
#
# Obiettivo: Controllare se le nostre classi (0 e 1) sono
# bilanciate. Se abbiamo troppi "1" e pochi "0", l'AI
# imparerà male, diventerà "prevenuto".
# ==============================================================================

print("\n\n--- 6. Controllo il bilanciamento delle classi ---")
# ".value_counts()" conta quante righe ci sono per ogni classe
print(df_pulito['is_toxic'].value_counts())

# RISULTATO ATTESO:
# 1 (Tossico)    -> 20620 righe
# 0 (Non Tossico) -> 4163 righe
#
# Questo è un PROBLEMA ENORME!
# È come insegnare a un bambino a riconoscere le mele
# mostrandogli 20.000 foto di arance e solo 4.000 di mele.
# Imparerà a dire "arancia" quasi sempre.

In [ ]:
print("\n--- 7. Bilanciamento con 'Undersampling' ---")
# SOLUZIONE: Sotto-campionamento (Undersampling)
# È la tecnica più semplice: "buttare via" in modo casuale
# molti esempi della classe maggioritaria (i "Tossici")
# fino a quando non ne abbiamo quanti quelli della classe minoritaria.

# Separiamo i dati in due tabelle
df_tossici = df_pulito[df_pulito['is_toxic'] == 1]
df_non_tossici = df_pulito[df_pulito['is_toxic'] == 0]

# Troviamo la dimensione della classe più piccola (i non tossici)
dimensione_minoritaria = len(df_non_tossici) # circa 4163

# Ora prendiamo un campione casuale dalla classe "tossici"
# grande esattamente come la classe "non tossici"
df_tossici_campione = df_tossici.sample(dimensione_minoritaria, random_state=42)

# Infine, uniamo le due tabelle (i non tossici + il campione di tossici)
# per creare il nostro dataset finale e bilanciato.
df_bilanciato = pd.concat([df_tossici_campione, df_non_tossici])

print("\nControllo il bilanciamento del NUOVO dataset:")
# ".value_counts()" ora dovrebbe dare numeri uguali (o molto simili)
print(df_bilanciato['is_toxic'].value_counts())
print("--- Dati perfettamente bilanciati! ---")

In [ ]:
# ==============================================================================
# ⚙️ BLOCCO 4: TRASFORMAZIONE (Da Testo a Numeri)
#
# Obiettivo: Trasformare le frasi (testo) in vettori (numeri).
# L'AI (LogisticRegression) non capisce le parole,
# capisce solo i numeri.
# ==============================================================================

print("\n\n--- 8. creiamo i vettori per ogni parola ---")

# Questa è la funzione "magica" che fa la media dei vettori W2V

def crea_vettore_frase(frase, modello):
    # Assicuriamoci che l'input sia testo (str)
    frase_str = str(frase)
    # 1. Pulizia e scomposizione
    tokens = frase_str.lower().split()
    vettori_parole = []
    
    # 2. per ogni token nella lista di token
    for token in tokens: 
        if token in modello:#se il modello conosce il token
            vettori_parole.append(modello[token]) #allora costruiamo la rappresentazione
            
    # 3. CONTROLLO (FUORI DAL FOR): SE NON CI SONO VETTORI = SE IL MODELLO NON CONOSCE NESSUNA DELLE PAROLE NELLA FRASE
    if not vettori_parole:
        # Se la lista è rimasta vuota, restituiamo il silenzio (zeri)
        return np.zeros(modello.vector_size)
    
    # 4. MEDIA (FUORI DAL FOR): Se invece ci sono vettori, facciamo la media
    return np.mean(vettori_parole, axis=0)

In [ ]:
print("\n\n--- 9. Applichiamo la funzione ---")


df_bilanciato['vettore'] = df_bilanciato['comment_text'].apply(
lambda frase: crea_vettore_frase(frase, modello_w2v)
)

df_bilanciato.head()

In [ ]:
# ==============================================================================
# 🎓 BLOCCO 5: ADDESTRAMENTO E TEST (Training & Evaluation)
#
# Obiettivo: Dividere i dati in "compiti per casa" (training) e
# "esame finale" (test), addestrare l'AI e vedere che voto prende.
# ==============================================================================

print("\n\n--- 10. Preparo i dati finali: X e y ---")
# X = I dati di input (i nostri vettori)
# y = Le risposte (le etichette 0 o 1)

# "np.stack" impila tutti i singoli vettori (dalla colonna 'vettore')
# in un'unica grande matrice (un "foglio" di numeri)
X = np.stack(df_bilanciato['vettore'].values)

# "y" è la colonna delle risposte (0 o 1)
y = df_bilanciato['is_toxic'].values

print(f"Forma di X (input): {X.shape}") # (numero_righe, 300)
print(f"Forma di y (risposte): {y.shape}") # (numero_righe,)

In [ ]:
print("\n--- 11. Divisione in Training Set e Test Set ---")
# Dobbiamo "nascondere" alcuni dati all'AI per l'esame finale.
# "test_size=0.2" -> Metti da parte il 20% dei dati per l'esame (Test Set)
# L'80% rimanente sarà usato per l'addestramento (Training Set)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Dati di addestramento (X_train): {X_train.shape[0]} esempi")
print(f"Dati di test (X_test): {X_test.shape[0]} esempi")

In [ ]:
print("\n--- 12. Addestramento del Modello (LogisticRegression) ---")
# Creiamo il "cervello" AI. La Regressione Logistica è un
# classificatore semplice, veloce e ottimo per i principianti.
classificatore = LogisticRegression(max_iter=1000) # max_iter=1000 per sicurezza

# "fit" è il comando "IMPARA!"
# L'AI guarda tutti gli esempi di training (X_train) e le loro
# risposte (y_train) e cerca di trovare la "formula matematica"
# per distinguere i vettori "0" dai vettori "1".
print("Addestramento in corso...")
classificatore.fit(X_train, y_train)
print("--- Addestramento completato! ---")

In [ ]:
print("\n--- 13. Valutazione sul Test Set (L'ESAME FINALE) ---")
# Ora usiamo l'AI addestrato per "predire" le risposte
# del Test Set (X_test), che non ha MAI visto prima.
predizioni = classificatore.predict(X_test)

# Calcoliamo l'accuratezza: quante risposte ha indovinato?
accuratezza = accuracy_score(y_test, predizioni)

print(f"RISULTATO ESAME: Accuratezza sul Test Set = {accuratezza * 100:.2f}%")

# L'accuratezza dovrebbe essere buona (es. > 85%)!
# Significa che il nostro modello ha imparato a distinguere
# i commenti tossici da quelli non tossici basandosi
# SOLO sulla media dei vettori Word2Vec.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# ==============================================================================
# 🔬 BLOCCO 3: L'ESPERIMENTO (Dimostrazione del Limite)
#
# Creeremo due coppie di frasi.
# 1. Frasi con significato OPPOSTO, ma stesse parole.
# 2. Frasi con significato OPPOSTO, per colpa di una negazione.
#
# E vedremo cosa "pensa" il nostro modello W2V.
# ==============================================================================

print("\n\n--- 5. ESPERIMENTO 1: L'ORDINE DELLE PAROLE ---")
print("Confrontiamo due frasi con significato opposto ma parole identiche.")

frase_A = "the dog bit the man" # "il cane ha morso l'uomo"
frase_B = "the man bit the dog" # "l'uomo ha morso il cane"

# 1. Trasformiamo le due frasi in vettori (facendo la media)
vettore_A = crea_vettore_frase(frase_A, modello_w2v)
vettore_B = crea_vettore_frase(frase_B, modello_w2v)

# 2. Calcoliamo la similarità tra i due vettori
# (Dobbiamo "rimodellarli" in 2D per farli piacere a scikit-learn)
vettore_A_2d = vettore_A.reshape(1, -1)
vettore_B_2d = vettore_B.reshape(1, -1)

similarita_AB = cosine_similarity(vettore_A_2d, vettore_B_2d)

print(f"\nFrase A: '{frase_A}'")
print(f"Frase B: '{frase_B}'")
print(f"Similarità calcolata da Word2Vec (media): {similarita_AB[0][0]:.4f}")

print("\nRISULTATO: La similarità è 1.0 (o quasi).")
print("Per il nostro modello W2V, le due frasi sono IDENTICHE.")


print("\n\n--- 6. ESPERIMENTO 2: LA NEGAZIONE ---")
print("Confrontiamo una frase positiva con la sua negazione.")

frase_C = "this movie is good"       # "questo film è bello"
frase_D = "this movie is not good"   # "questo film NON è bello"

# 1. Trasformiamo le frasi in vettori
vettore_C = crea_vettore_frase(frase_C, modello_w2v)
vettore_D = crea_vettore_frase(frase_D, modello_w2v)

# 2. Calcoliamo la similarità
vettore_C_2d = vettore_C.reshape(1, -1)
vettore_D_2d = vettore_D.reshape(1, -1)

similarita_CD = cosine_similarity(vettore_C_2d, vettore_D_2d)

print(f"\nFrase C: '{frase_C}'")
print(f"Frase D: '{frase_D}'")
print(f"Similarità calcolata da Word2Vec (media): {similarita_CD[0][0]:.4f}")

## ESERCIZIO IN SEMI-AUTONOMIA: 

Prendete l'esercitazione sulla sentiment analysis vista in classe insieme nell'ultima lezione. 
- Importate il dataset 
- Create un processo di pre processing sul testo giustificando ogni singolo step (esempio: faccio la rimozione delle stopword. Perchè...)
- Costruite una colonna nel vostro dataframe che contenga il testo dopo il pre processing
- Applicate il processo di rappresentazione dei testi sfruttando W2V (processo visto oggi) creando una colonna che contenga per ogni riga la rappresentazione W2V Media di ogni documento
- utilizzate la rappresentazione ottenuta per addestrare una regressione

Il codice di ognuno di questi step deve essere preceduto dal ragionamento effettuato.